# Prompt-injection 16-model study - Kaggle training notebook (thin wrapper)

This notebook only *calls* the repository modules; all logic lives in src/.

**Before running**
1. Add the repository as a Kaggle *Dataset* (zip of this repo without outputs/, data/) or git clone it in the first cell (Internet must be enabled for cloning / model downloads).
2. Add the frozen Phase 3.5 split package (kaggle_package_v1_leakfree.zip contents) as a Kaggle Dataset -> mounted read-only under /kaggle/input/....
3. Optional: add an HF_TOKEN Kaggle **Secret** (needed only for gated checkpoints such as Llama-3.1). Never paste tokens into cells.
4. Accelerator: GPU T4 x2 / P100 for encoders; for 7B/8B QLoRA a 16 GB GPU is the minimum.

**Rules enforced by the code**: /kaggle/input is read-only, every artifact goes to /kaggle/working/outputs, the data manifest must be frozen + leakage-clean, no Kaggle API / dataset publishing / submissions are performed.

In [ ]:
# 1. Repository ----------------------------------------------------------------
import os, subprocess, sys, pathlib
REPO_DIR = pathlib.Path('/kaggle/working/repo')
REPO_URL = os.environ.get('REPO_URL', '')   # set via Kaggle Secret or edit; leave empty if repo is mounted as dataset
REPO_INPUT = next((p for p in pathlib.Path('/kaggle/input').glob('*/src/models/registry.py')), None)
if REPO_INPUT is not None:
    import shutil
    shutil.copytree(REPO_INPUT.parents[2], REPO_DIR, dirs_exist_ok=True)
elif REPO_URL and not REPO_DIR.exists():
    subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, str(REPO_DIR)], check=True)
assert (REPO_DIR / 'src').exists(), 'repository not found: mount it as a dataset or set REPO_URL'
os.chdir(REPO_DIR); sys.path.insert(0, str(REPO_DIR))
print('repo at', REPO_DIR)

In [ ]:
# 2. Pinned dependencies (Kaggle image already ships torch) ----------------------
%pip install -q -r requirements-kaggle.txt

In [ ]:
# 3. Environment detection ------------------------------------------------------
from src.kaggle.env import detect, find_kaggle_splits_dir
env = detect()
print(env.to_dict())
SPLITS_DIR = find_kaggle_splits_dir()
assert SPLITS_DIR is not None, 'mount the kaggle_package_v1_leakfree dataset (needs manifest.json + train.parquet)'
print('splits:', SPLITS_DIR)

In [ ]:
# 4. Leakage gate (fails loudly if the manifest is not frozen / clean) -------------
from src.data.phase35 import load_frozen_manifest
manifest = load_frozen_manifest(SPLITS_DIR)
print('version', manifest['version'], '| leakage passed:', manifest['leakage_passed'])
print({k: v['_total'] for k, v in manifest['final_counts'].items()})

In [ ]:
# 5. Choose models / seeds for THIS session -------------------------------------
# One session rarely fits all 16 models x 3 seeds; the manifest in /kaggle/working/outputs/sweep
# makes the sweep resumable: re-run the notebook (or attach the previous output as input and copy
# it back to /kaggle/working/outputs) to continue where it stopped.
MODELS = ['textcnn', 'distilbert']          # e.g. ['roberta', 'deberta_v3'] on a GPU session
SEEDS = [42]                                # production: [42, 43, 44]
BUDGET_MIN = 60 * 8                         # stop launching jobs after 8h (Kaggle limit is 12h GPU / 9h TPU)
DEBUG = True                                # True -> configs/kaggle_debug.yaml row caps

In [ ]:
# 6. Run the resumable sweep ----------------------------------------------------
from src.training import sweep
cfg = 'configs/kaggle_debug.yaml' if DEBUG else 'configs/experiment.yaml'
argv = ['--config', cfg, '--models', *MODELS, '--seeds', *map(str, SEEDS),
        '--splits-dir', str(SPLITS_DIR), '--runs-dir', '/kaggle/working/outputs/runs',
        '--manifest', '/kaggle/working/outputs/sweep/manifest.json', '--budget-minutes', str(BUDGET_MIN)]
sweep.main(argv)

In [ ]:
# 7. Tables + figures from saved predictions only --------------------------------
from src.eval import tables, plots
tables.main(['--runs', '/kaggle/working/outputs/runs', '--out', '/kaggle/working/outputs/tables', '--n-boot', '500'])
plots.main(['--runs', '/kaggle/working/outputs/runs', '--out', '/kaggle/working/outputs/figures', '--splits-dir', str(SPLITS_DIR)])

In [ ]:
# 8. Registry status derived from artifacts -------------------------------------
from src.models import registry
registry.main(['--runs-dir', '/kaggle/working/outputs/runs', '--export', '/kaggle/working/outputs/registry.json'])
# Download /kaggle/working/outputs (Output tab) and copy into the repo's outputs/ to update docs locally.